In [3]:
import pandas as pd
import re
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import statsmodels.api as sm
import numpy as np
from stargazer.stargazer import Stargazer



In [4]:
df = pd.read_parquet(r"C:\Users\Mateus Monteleone\Projects\ic\data\binary_model_table.parquet")


In [5]:
df.head()

,full_text,clean_text,unsupervised_sentiment,mentioned_candidates,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes,is_economic
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0,[lula],1,0,0,0,0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0,[lula],1,0,0,0,0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0,[lula],1,0,0,0,0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0,[lula],1,0,0,0,0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0,[lula],1,0,0,0,0


In [4]:
print(df.columns)

Index(['full_text', 'clean_text', 'unsupervised_sentiment',
       'mentioned_candidates', 'mentions_lula', 'mentions_bolsonaro',
       'mentions_tebet', 'mentions_ciro_gomes', 'is_economic'],
      dtype='object')


In [5]:
len(df)

190300

In [6]:
df["mentions_bolsonaro_and_economic"] = (
    (df["mentions_bolsonaro"] == 1) & (df["is_economic"] == 1)
).astype(int)

df["mentions_lula_and_economic"] = (
    (df["mentions_lula"] == 1) & (df["is_economic"] == 1)
).astype(int)

df["mentions_tebet_and_economic"] = (
    (df["mentions_tebet"] == 1) & (df["is_economic"] == 1)
).astype(int)

df["mentions_ciro_and_economic"] = (
    (df["mentions_ciro_gomes"] == 1) & (df["is_economic"] == 1)
).astype(int)

df[
    [
        "full_text",
        "clean_text",
        "unsupervised_sentiment",
        "is_economic",
        "mentions_lula",
        "mentions_bolsonaro",
        "mentions_tebet",
        "mentions_ciro_gomes",
        "mentions_bolsonaro_and_economic",
        "mentions_lula_and_economic",
        "mentions_tebet_and_economic",
        "mentions_ciro_and_economic",
    ]
].head()


,full_text,clean_text,unsupervised_sentiment,is_economic,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes,mentions_bolsonaro_and_economic,mentions_lula_and_economic,mentions_tebet_and_economic,mentions_ciro_and_economic
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0,0,1,0,0,0,0,0,0,0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0,0,1,0,0,0,0,0,0,0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0,0,1,0,0,0,0,0,0,0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0,0,1,0,0,0,0,0,0,0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0,0,1,0,0,0,0,0,0,0


In [7]:
binary_columns = [
    "is_economic",
    "mentions_lula",
    "mentions_bolsonaro",
    "mentions_tebet",
    "mentions_ciro_gomes",
    "mentions_lula_and_economic",
    "mentions_bolsonaro_and_economic",
    "mentions_tebet_and_economic",
    "mentions_ciro_and_economic",
]

for col in binary_columns:
    print(f"\n{col}")
    print(df[col].value_counts().sort_index())



is_economic
is_economic
0    186656
1      3644
Name: count, dtype: int64

mentions_lula
mentions_lula
0     71326
1    118974
Name: count, dtype: int64

mentions_bolsonaro
mentions_bolsonaro
0    114654
1     75646
Name: count, dtype: int64

mentions_tebet
mentions_tebet
0    186161
1      4139
Name: count, dtype: int64

mentions_ciro_gomes
mentions_ciro_gomes
0    170402
1     19898
Name: count, dtype: int64

mentions_lula_and_economic
mentions_lula_and_economic
0    188136
1      2164
Name: count, dtype: int64

mentions_bolsonaro_and_economic
mentions_bolsonaro_and_economic
0    188649
1      1651
Name: count, dtype: int64

mentions_tebet_and_economic
mentions_tebet_and_economic
0    190258
1        42
Name: count, dtype: int64

mentions_ciro_and_economic
mentions_ciro_and_economic
0    189882
1       418
Name: count, dtype: int64


In [8]:
X = df[binary_columns].copy()
y = df["unsupervised_sentiment"].copy()

In [9]:
X.head()

,is_economic,mentions_lula,mentions_bolsonaro,mentions_tebet,mentions_ciro_gomes,mentions_lula_and_economic,mentions_bolsonaro_and_economic,mentions_tebet_and_economic,mentions_ciro_and_economic
0,0,1,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0
4,0,1,0,0,0,0,0,0,0


In [22]:
X_sm = sm.add_constant(X)

# Modelo logit para sentimento positivo (y) binaria; x binaria p/ presidente_is_economic

logit_pos = sm.Logit((y == 1).astype(int), X_sm).fit()
result_positive = logit_pos
print("\nPOSITIVE MODEL")
print(result_positive.summary())

Optimization terminated successfully.
         Current function value: 0.639984
         Iterations 5

POSITIVE MODEL
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190290
Method:                               MLE   Df Model:                            9
Date:                    Wed, 03 Jun 2026   Pseudo R-squ.:                 0.02274
Time:                            18:08:32   Log-Likelihood:            -1.2179e+05
converged:                           True   LL-Null:                   -1.2462e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const             

In [11]:
stargazer = Stargazer([result_positive])
stargazer.title("Logistic Regression Results")
stargazer

In [17]:
print("\nOdds Ratios - Positive")
print(np.exp(result_positive.params))


Odds Ratios - Positive
const                              0.971772
is_economic                        0.634347
mentions_lula                      0.758216
mentions_bolsonaro                 0.403757
mentions_tebet                     0.675073
mentions_ciro_gomes                0.919585
mentions_lula_and_economic         1.028596
mentions_bolsonaro_and_economic    1.181816
mentions_tebet_and_economic        0.681353
mentions_ciro_and_economic         2.112021
dtype: float64


In [ ]:
# Modelo logit binario para sentimento neutro (y) binaria; x binaria p/ presidentes + is_economic

logit_neu = sm.Logit((y == 0).astype(int), X_sm).fit()
result_neutral = logit_neu
print("Neutral Model: ")
print(result_neutral.summary())

Optimization terminated successfully.
         Current function value: 0.584539
         Iterations 7
Neutral Model: 
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190290
Method:                               MLE   Df Model:                            9
Date:                    Wed, 06 May 2026   Pseudo R-squ.:                 0.01138
Time:                            17:50:54   Log-Likelihood:            -1.1124e+05
converged:                           True   LL-Null:                   -1.1252e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const             

In [19]:
print("\nOdds Ratios - Neutral")
print(np.exp(result_neutral.params))



Odds Ratios - Neutral
const                              0.369472
is_economic                        0.401909
mentions_lula                      1.153416
mentions_bolsonaro                 1.045043
mentions_tebet                     2.348270
mentions_ciro_gomes                0.443108
mentions_lula_and_economic         0.856763
mentions_bolsonaro_and_economic    1.172807
mentions_tebet_and_economic        2.395031
mentions_ciro_and_economic         0.816866
dtype: float64


In [20]:
# Modelo logit para sentimento negativo (y) binaria; x binaria p/ presidente_is_economic

logit_neg = sm.Logit((y == -1).astype(int), X_sm).fit()
result_negative = logit_neg
print("Negative Model: ")
print(result_negative.summary())

Optimization terminated successfully.
         Current function value: 0.634887
         Iterations 5
Negative Model: 
                             Logit Regression Results                             
Dep. Variable:     unsupervised_sentiment   No. Observations:               190300
Model:                              Logit   Df Residuals:                   190290
Method:                               MLE   Df Model:                            9
Date:                    Wed, 06 May 2026   Pseudo R-squ.:                 0.02769
Time:                            17:51:05   Log-Likelihood:            -1.2082e+05
converged:                           True   LL-Null:                   -1.2426e+05
Covariance Type:                nonrobust   LLR p-value:                     0.000
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
const            

In [21]:
print("\nOdds Ratios - Negative")
print(np.exp(result_negative.params))


Odds Ratios - Negative
const                              0.344377
is_economic                        2.770391
mentions_lula                      1.111667
mentions_bolsonaro                 2.266836
mentions_tebet                     0.720958
mentions_ciro_gomes                1.887746
mentions_lula_and_economic         1.132793
mentions_bolsonaro_and_economic    0.743492
mentions_tebet_and_economic        1.050639
mentions_ciro_and_economic         0.393811
dtype: float64
